# ESMFold interactive folding

Run this notebook inside the VS Code Dev Container. Edit the sequence cell, then run the cells top to bottom.

In [1]:
import pathlib
import time
import pandas as pd
import esm
import torch
from   tqdm import tqdm

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

torch 2.7.1+cu128
cuda available True
gpu NVIDIA RTX PRO 2000 Blackwell Generation Laptop GPU


In [2]:
df = pd.read_csv("./data/1a3n_chainA_samples_t1.00.csv")
df

,id,sequence
0,1a3n_native_seq,VLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHF...
1,sample1_t1.00_0.496,PLTARDKAAVKKVWGATGAAAPALGAEAFQHLFKTYPEAKEYFKNF...
2,sample2_t1.00_0.504,ALSSQDKQNVSAVWGAVGGQAGALGAEAFQCMFSKYPGAKAYFPNF...
3,sample3_t1.00_0.461,TLTAADRDAVAAALGAVGDDAGALGAEALQNMFTTYPPTRAYFGNF...
4,sample4_t1.00_0.553,SLSTDDRRAVQAAFGSVGVNAGAWGAEALQCMFTQFPKTRSYFGQF...
5,sample5_t1.00_0.511,SLSATDKANVKAAIGKLGDKGGALGAEALTKTFAKVPEARQYFKNF...
6,sample6_t1.00_0.475,SLSEEDRANVRGALGAVGDKAGTYGAKAFQYAFKTYPETAAFFGQF...
7,sample7_t1.00_0.468,NLTDDDMAAVKAVWGKTGGQGGALGAQALQKSFKAFPPTCEYFPFM...
8,sample8_t1.00_0.468,PLSAEDKANVSAAWSAVGAKAPELGAKALQNTFDKYPQAKQYFPNF...
9,sample9_t1.00_0.496,GLSDTDRANVRAAWGVVGGSGGRLGAEALQCLFSSFPAARAFFPNM...


In [3]:
# Paste your protein sequence here. For multimers, separate chains with ':'
#sequence = "VLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR"
#output_path = pathlib.Path("output_notebook.pdb")

num_recycles = 3
chunk_size = 32
precision = "bf16"  # Good default for this 8 GB Blackwell GPU.

In [4]:
start = time.time()
model = esm.pretrained.esmfold_v1().eval()

if torch.cuda.is_available():
    if precision == "bf16":
        model = model.bfloat16()
    elif precision == "fp16":
        model = model.half()
    model = model.cuda()
else:
    model = model.cpu()

model.set_chunk_size(chunk_size)
print(f"model loaded in {time.time() - start:.1f}s")

/opt/micromamba/envs/pepgen/lib/python3.9/site-packages/deepspeed/runtime/zero/linear.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, weight, bias=None):
/opt/micromamba/envs/pepgen/lib/python3.9/site-packages/deepspeed/runtime/zero/linear.py:70: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):


model loaded in 39.3s


In [5]:
start = time.time()

with torch.no_grad():
    for sequence, id in tqdm(zip(df["sequence"].values, df["id"].values), total=len(df)):
        output_path = pathlib.Path(f"./output/{id}.pdb")
        result = model.infer(sequence, num_recycles=num_recycles)

        # NumPy cannot export BF16 tensors, so cast floating outputs before PDB conversion.
        result = {
            key: value.float() if torch.is_tensor(value) and torch.is_floating_point(value) else value
            for key, value in result.items()
        }

        pdb = model.output_to_pdb(result)[0]
        output_path.write_text(pdb)
        print(f"wrote {output_path.resolve()}")
print(f"modeling in {time.time() - start:.1f}s")

  0%|          | 0/11 [00:00<?, ?it/s]

/opt/micromamba/envs/pepgen/lib/python3.9/site-packages/openfold/model/primitives.py:201: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/opt/micromamba/envs/pepgen/lib/python3.9/site-packages/openfold/model/primitives.py:233: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
  9%|▉         | 1/11 [00:02<00:27,  2.78s/it]

wrote /workspace/esmfold-notebook/4-folding/output/1a3n_native_seq.pdb


 18%|█▊        | 2/11 [00:05<00:22,  2.49s/it]

wrote /workspace/esmfold-notebook/4-folding/output/sample1_t1.00_0.496.pdb


...

wrote /workspace/esmfold-notebook/4-folding/output/sample8_t1.00_0.468.pdb


 91%|█████████ | 10/11 [00:20<00:02,  2.11s/it]

wrote /workspace/esmfold-notebook/4-folding/output/sample9_t1.00_0.496.pdb


100%|██████████| 11/11 [00:22<00:00,  2.06s/it]

wrote /workspace/esmfold-notebook/4-folding/output/sample10_t1.00_0.504.pdb
modeling in 22.6s
